# 32 – DTO Patterns for Clean APIs

## Learning objectives

- Use `BaseDTO`, `DTOConfig`, and specialized `CreateDTO` / `ReadDTO` / `UpdateDTO` from the framework.
- Compare those helpers with Litestar-native `PydanticDTO` / `DataclassDTO` patterns.
- Keep sensitive model fields off the wire with explicit excludes.

## About this topic

DTOs define the contract your HTTP or RPC layer exposes. Different shapes apply to create, read, and partial update flows. Litestar can enforce renaming strategies (for example camelCase) at the edge; the core `BaseDTO` offers a lightweight, framework-agnostic starting point.

In [ ]:
from pydantic import Field, SecretStr

from agentic_assistants.core.base_models import AgenticEntity
from agentic_assistants.core.dto import CreateDTO, DTOConfig, ReadDTO, UpdateDTO

class ApiUser(AgenticEntity):
    email: str = Field(min_length=3)
    display_name: str
    password_hash: SecretStr = SecretStr("unset")

user = ApiUser(email="ada@example.com", display_name="Ada", password_hash=SecretStr("secret"))

class UserRead(ReadDTO[ApiUser]):
    class Config(DTOConfig):
        exclude = {"password_hash"}

class UserCreate(CreateDTO[ApiUser]):
    class Config(DTOConfig):
        exclude = CreateDTO.Config.exclude | {"password_hash"}

print("read dto:", UserRead.from_model(user))
print("create roundtrip fields:", UserCreate.to_model({"email": "bob@ex.com", "display_name": "Bob"}).display_name)

In [ ]:
from pydantic import Field, SecretStr

from agentic_assistants.core.base_models import AgenticEntity
from agentic_assistants.core.dto import ReadDTO, UpdateDTO

class Profile(AgenticEntity):
    handle: str
    bio: str = ""
    internal_score: float = 0.0
    password_hash: SecretStr = SecretStr("x")

class ProfilePublic(ReadDTO[Profile]):
    class Config:
        exclude = {"password_hash", "internal_score"}

class ProfilePatch(UpdateDTO[Profile]):
    class Config:
        exclude = UpdateDTO.Config.exclude | {"password_hash", "internal_score"}

p = Profile(handle="coder", bio="likes graphs")
print("public:", ProfilePublic.from_model(p))
patched = ProfilePatch.to_model({"handle": "coder", "bio": "likes LangGraph"})
print("patch bio:", patched.bio)

In [ ]:
from pathlib import Path
import importlib.util

def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd if (cwd / "pyproject.toml").exists() else cwd.parent

path = _repo_root() / "examples/library/litestar_patterns/dto_boundaries.py"
spec = importlib.util.spec_from_file_location("ls_dto", path)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
client = mod.TestClient(mod.create_app())
resp = client.post("/users", json={"email": "ada@ex.com", "displayName": "Ada"})
print(resp.status_code, resp.json())

## Exercises and next steps

1. Add `rename_strategy="camel"` to a custom `ReadDTO` and verify keys in `from_model`.
2. Extend `ApiUser` with `roles: list[str]` and decide which DTOs should expose it.
3. Add a failing test that proves `password_hash` never appears in `UserRead.from_model` output.